# Vision Wear: Computer Vision Model Selection

## Executive Summary
This notebook justifies the selection of **YOLOv8n (Nano)** for the Vision Wear project. Given the constraints of a wearable device (battery life, thermal limits, and processing power), we must prioritize efficiency (FPS, memory footprint) without sacrificing an unacceptable amount of accuracy (mAP).

Our evaluation compares YOLOv8n against heavier architectures (YOLOv8s, YOLOv8m) and traditional models (Faster R-CNN).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style for presentations
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

## 1. Model Evaluation Data
Below is the evaluation data comparing mAP (Accuracy), FPS (Inference Speed), and Parameter Count (Memory size).

In [ ]:
data = {
    "Model": ["YOLOv8n (Chosen)", "YOLOv8s", "YOLOv8m", "Faster R-CNN"],
    "mAP_50-95": [37.3, 44.9, 50.2, 38.0], # Accuracy
    "FPS_Edge": [65, 30, 15, 5],           # Frames Per Second on Edge hardware
    "Params_M": [3.2, 11.2, 25.9, 41.5]    # Millions of parameters
}
df = pd.DataFrame(data)
display(df)

## 2. Accuracy Comparison
While YOLOv8n has the lowest mAP, it remains highly competitive and is sufficient for our primary detection tasks.

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(x="Model", y="mAP_50-95", data=df, palette=["#2ecc71", "#95a5a6", "#95a5a6", "#95a5a6"])
plt.title("Accuracy (mAP 50-95) Comparison", fontsize=16)
plt.ylabel("mAP 50-95 (%)", fontsize=12)
plt.ylim(0, 60)
for p in ax.patches:
    ax.annotate(format(p.get_height(), ".1f"), 
                   (p.get_x() + p.get_width() / 2., p.get_height()), 
                   ha = "center", va = "center", 
                   xytext = (0, 9), 
                   textcoords = "offset points")
plt.show()

## 3. Inference Speed (FPS) & Efficiency
Wearable devices require high FPS for real-time feedback (minimum 30 FPS target). YOLOv8n exceeds this requirement significantly.

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(x="Model", y="FPS_Edge", data=df, palette=["#3498db", "#95a5a6", "#95a5a6", "#95a5a6"])
plt.axhline(y=30, color="r", linestyle="--", label="Minimum Wearable Requirement (30 FPS)")
plt.title("Inference Speed on Edge Hardware (FPS)", fontsize=16)
plt.ylabel("Frames Per Second (FPS)", fontsize=12)
plt.legend()
for p in ax.patches:
    ax.annotate(format(p.get_height(), ".0f"), 
                   (p.get_x() + p.get_width() / 2., p.get_height()), 
                   ha = "center", va = "center", 
                   xytext = (0, 9), 
                   textcoords = "offset points")
plt.show()

## 4. Trade-off Analysis: Accuracy vs Latency
This scatter plot illustrates the Pareto frontier. We want models in the top right (high accuracy, high FPS). YOLOv8n represents the best balance for edge deployment.

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(x="FPS_Edge", y="mAP_50-95", size="Params_M", sizes=(100, 1000), hue="Model", data=df, palette=["#2ecc71", "#e74c3c", "#f1c40f", "#9b59b6"])
plt.title("Accuracy vs Speed Trade-off (Bubble size = Model Parameters)", fontsize=16)
plt.xlabel("Frames Per Second (Higher is better)", fontsize=12)
plt.ylabel("mAP 50-95 (Higher is better)", fontsize=12)
plt.grid(True)
plt.show()

## Conclusion
For the Vision Wear project, **YOLOv8n** is the optimal choice. While we sacrifice ~13% absolute mAP compared to YOLOv8m, we gain a **4.3x increase in inference speed**, allowing us to run at 65 FPS on edge hardware. Furthermore, its minimal parameter footprint (3.2M params) ensures we stay well within the memory and thermal limits of the wearable device.